# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema at:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset and inspect metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata as attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Keywords: {dataset.metadata.keywords}")

## 2. Data Overview

Review available record sets and their fields. Each entity is referenced by its Croissant `@id` for clarity and reproducibility.


In [ ]:
# List all available record sets by their @id
print("Available Record Sets (by @id):\n-------------------------------")
for record_set in dataset.record_sets:
    print(f"@id: {record_set['@id']}, name: {record_set.get('name','')} ")

# For demonstration, we'll select all record sets for further exploration
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

# For each record set, show its fields
for record_set in dataset.record_sets:
    print(f"\nFields for Record Set @id: {record_set['@id']}")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            field_id = field['@id']
            name = field.get('name', '')
        else:
            field_id = field
            name = ''
        print(f"    Field @id: {field_id} name: {name}")

## 3. Data Extraction
Load data from each record set into DataFrames, referencing `@id`s for record sets and fields.


In [ ]:
# Load data from all available record sets
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for Record Set: {record_set_id}")
            print(f"Columns: {df.columns.tolist()}")
        else:
            print(f"No records found for Record Set: {record_set_id}")
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

# Use the main record set for the protein records (replace with actual id from previous cell output as needed)
# For this dataset, the main table typically has an id like 'https://api.app.sen.science/frontiers/7154140/TabularData-1'
# We'll try to auto-select the largest one
if dataframes:
    main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k]))
    print(f"\nUsing main record set: {main_record_set_id}\nColumns: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)

We'll clean and explore the main record set. Operations are performed using fields referenced by their Croissant `@id`.


In [ ]:
# For demonstration, select a numeric field `cr:mw` (Molecular Weight) if available
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Try common numeric Croissant field ids (please adjust as necessary from data overview output):
    candidate_numeric_fields = ['cr:mw', 'mw', '@mw', 'MolecularWeight']
    numeric_field = None
    for field in candidate_numeric_fields:
        if field in df.columns:
            numeric_field = field
            break
    if numeric_field is None:
        # fallback to first numeric field detected
        numeric_field = df.select_dtypes(include='number').columns[0]
    print(f"Selected numeric field for analysis: {numeric_field}")

    threshold = df[numeric_field].quantile(0.75)  # use 75th percentile as example
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by a categorical field, e.g., 'cr:sample' or 'Sample', typical for proteomics
    candidate_group_fields = ['cr:sample', 'Sample', '@sample', 'cr:accession', 'accession']
    group_field = None
    for field in candidate_group_fields:
        if field in df.columns:
            group_field = field
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
else:
    print("No main record set for EDA.")

## 5. Visualization
Visualize distributions and relationships between fields, referencing all columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None:
    # Histogram of the main numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=40, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If we have a group field, plot boxplot
    if group_field:
        # Use top 10 frequent groups for clarity
        top_groups = df[group_field].value_counts().index[:10]
        plt.figure(figsize=(12,5))
        sns.boxplot(x=df[group_field][df[group_field].isin(top_groups)], y=df[numeric_field][df[group_field].isin(top_groups)])
        plt.title(f'{numeric_field} by {group_field} (Top 10 groups)')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

We loaded the FAIR² mass spectrometry dataset using its Croissant schema and explored its contents with `mlcroissant`. By referencing record sets and fields using their Croissant `@id`s, we ensured reproducibility and clarity. We performed basic filtering, normalization, grouping, and visualization on the main records table. This workflow can be extended for domain-specific analysis, downstream preprocessing, or integration in machine learning pipelines.
